# 🎙 말.통 더빙 — 원클릭 (Colab 무료 GPU)

유튜브 영상을 **원래 목소리 그대로** 영어·중국어·일본어·스페인어·독일어·프랑스어로 더빙해 **영상(mp4)·음성(mp3)·자막(srt)** 을 zip 으로 내려받아요.

**쓰는 법 — 딱 한 번만 누르면 돼요**
1. 아래 칸에 유튜브 링크를 넣고 언어·목소리를 고른 뒤, 맨 아래 **동의**에 체크
2. 칸 왼쪽의 **▶** 를 한 번 누르기 → 설치·더빙·내려받기까지 알아서 진행돼요

- 이 노트북은 열면 자동으로 **T4 GPU** 를 씁니다 (안 되면 `런타임 → 런타임 유형 변경 → T4 GPU`).
- 처음 실행은 설치·모델 내려받기로 3~5분 더 걸리고, 두 번째부터는 바로 시작해요.
- 유튜브가 Colab 에서의 내려받기를 막으면(봇 확인), 그 영상 파일을 올리라는 창이 떠요.
- Colab 판은 자막 검토 없이 끝까지 만들어요. 자막을 고친 뒤 만들고 싶으면 이 Mac 의 바탕화면 **말통 더빙** 앱을 쓰세요.

⚠️ **본인 목소리이거나 허락받은 목소리만** 쓰세요. 공개할 때는 "AI 음성"이라고 밝혀 주세요.
코드: [github.com/herwoojin/multi_chat/dubbing](https://github.com/herwoojin/multi_chat/tree/main/dubbing)

In [ ]:
#@title 🎙 말.통 더빙 — 칸을 채우고 ▶ 를 한 번 누르세요 { display-mode: "form" }
#@markdown ### 1. 유튜브 링크 (비워 두면 내 영상 파일을 올리라고 물어봐요)
youtube_url = ""  #@param {type:"string"}
#@markdown ### 2. 더빙할 언어
English = True  #@param {type:"boolean"}
Chinese = True  #@param {type:"boolean"}
Japanese = True  #@param {type:"boolean"}
Spanish = True  #@param {type:"boolean"}
German = True  #@param {type:"boolean"}
French = True  #@param {type:"boolean"}
Korean = False  #@param {type:"boolean"}
#@markdown ### 3. 목소리 — 영상 속 목소리를 쓰거나, 내 목소리 파일(10~30초)을 올려요
voice = "영상 속 목소리"  #@param ["영상 속 목소리", "내 목소리 파일 올리기"]
#@markdown ### 4. 자주 나오는 단어 (선택 — 받아쓰기에서 비슷하게 들린 말을 이 표기로 고쳐요. 예: 제미나이, 앱시트, 깃허브)
glossary = ""  #@param {type:"string"}
#@markdown ### 5. 앞부분만 몇 분 (0 이면 전체 — 처음엔 1~2분으로 시험해 보세요)
first_minutes = 0  #@param {type:"number"}
#@markdown ### 6. 말투 (보통은 그대로)
emotion = 0.5  #@param {type:"slider", min:0.25, max:1, step:0.05}
calmness = 0.5  #@param {type:"slider", min:0.2, max:1, step:0.05}
#@markdown ---
#@markdown ### ✅ 본인 목소리이거나 허락받은 목소리예요. 공개할 때 "AI 음성"이라고 밝힐게요.
i_agree = False  #@param {type:"boolean"}

import os, subprocess
from pathlib import Path
from IPython.display import Video, display
from google.colab import files

REPO = 'https://github.com/herwoojin/multi_chat.git'
WORK, VENV = Path('/content/maltong'), Path('/content/dub-venv')
PY = VENV / 'bin' / 'python'
LANGS = [c for c, on in [('en', English), ('zh', Chinese), ('ja', Japanese), ('es', Spanish), ('de', German), ('fr', French), ('ko', Korean)] if on]

def step(msg):
    print(f'\n▶ {msg}', flush=True)

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

def upload(what):
    step(f'{what}을(를) 고르세요')
    name = next(iter(files.upload()))
    return str(Path('/content') / name)

if not i_agree:
    raise SystemExit('✋ 맨 아래 동의 칸에 체크한 뒤 다시 ▶ 를 눌러 주세요 (본인 또는 허락받은 목소리만).')
if not LANGS:
    raise SystemExit('✋ 더빙할 언어를 하나 이상 골라 주세요.')
gpu = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or '없음 — 런타임 → 런타임 유형 변경 → T4 GPU 를 고르면 몇 배 빨라요')

# 1) 코드 + 전용 Python 3.11 환경 — Colab 기본 패키지와 섞지 않아 torch·torchvision 충돌이 없다. 두 번째부터는 건너뜀
if WORK.exists():
    sh(f'git -C {WORK} pull -q --ff-only')
else:
    step('코드 받는 중…')
    sh(f'git clone -q --depth 1 {REPO} {WORK}')
if not (VENV / '.ok').exists():
    step('설치 중… (처음 한 번 2~4분)')
    sh('pip install -q uv')
    sh(f'uv venv -q -p 3.11 {VENV}')
    sh(f'uv pip install -q -p {PY} -r {WORK}/dubbing/requirements-gpu.txt')
    (VENV / '.ok').touch()

# 2) 영상 · 목소리
source = youtube_url.strip() or upload('더빙할 영상 파일')
voice_args = ['--voice', upload('내 목소리 파일(10~30초)')] if voice == '내 목소리 파일 올리기' else []

def run(src):
    cmd = [str(PY), '-m', 'dubber', src, '--langs', ','.join(LANGS), '--exaggeration', str(emotion),
           '--cfg', str(calmness), '--i-have-permission', *voice_args]
    if first_minutes and first_minutes > 0:
        cmd += ['--minutes', str(first_minutes)]
    if glossary.strip():
        cmd += ['--glossary', glossary.strip()]
    env = {**os.environ, 'PYTHONWARNINGS': 'ignore', 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, cwd=WORK / 'dubbing', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    log = []
    try:
        for line in p.stdout:
            log.append(line)
            if line.startswith(('[', '✅', '✖', '   ')):  # 진행 상황만 보여준다
                print(line, end='', flush=True)
    except KeyboardInterrupt:
        p.terminate()
        raise
    return p.wait(), ''.join(log)

step('더빙 시작 — 진행률과 남은 시간이 아래에 표시돼요')
code, log = run(source)
# 유튜브가 Colab 서버의 내려받기를 막으면(봇 확인 등) 음성 인식까지 못 간다 → 그 영상 파일로 다시
if code != 0 and source.startswith('http') and '음성 인식' not in log:
    print('\n⚠️ 유튜브가 Colab 에서의 내려받기를 막았어요. 내 컴퓨터에 있는 그 영상 파일을 올려 주세요.')
    source = upload('같은 영상 파일')
    code, log = run(source)
if code != 0:
    print(log[-3000:])
    raise SystemExit('✖ 더빙에 실패했어요. 위 로그를 확인해 주세요.')

# 3) 결과 — 첫 언어 영상을 미리 보여 주고 전체 zip 을 내려받는다
job = max((WORK / 'dubbing' / 'jobs').iterdir(), key=lambda d: d.stat().st_mtime)
zip_path = next(job.glob('*_dubbed.zip'))
preview = next(iter(sorted(job.glob('out/*/video_*.mp4'))), None)
if preview and preview.stat().st_size < 40 * 1048576:
    display(Video(str(preview), embed=True, width=480))
step(f'완료 ✅ {zip_path.name} ({zip_path.stat().st_size / 1048576:.1f}MB) 를 내려받아요')
files.download(str(zip_path))